# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @id, name, and available fields
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  name: {rs.get('name', 'N/A')}")
        print(f"  description: {rs.get('description', 'N/A')}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields @id:")
            for field in fields:
                print(f"    - {field['@id']}")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set @ids to extract
def get_record_set_ids(metadata):
    if hasattr(metadata, 'record_sets'):
        return [rs['@id'] for rs in metadata.record_sets]
    return []
record_set_ids = get_record_set_ids(metadata)
dataframes = {}
for record_set_id in record_set_ids:
    # Load records for each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet: {record_set_id}")
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# Demonstration: If any records sets found, print columns of the first one
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in RecordSet '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field in the first available record set
import numpy as np

if dataframes:
    df = next(iter(dataframes.values()))
    rs_id = next(iter(dataframes.keys()))
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field} from RecordSet {rs_id}")
        threshold = df[numeric_field].mean() if not df[numeric_field].mean() == 0 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Try grouping by a categorical field if available
        cat_fields = df.select_dtypes(include=[object]).columns.tolist()
        if cat_fields:
            group_field = cat_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} grouped by {group_field}")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes:
    df = next(iter(dataframes.values()))
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we demonstrated how to discover record sets, extract records, inspect fields, and perform minimal EDA and visualization using the `mlcroissant` library on the FAIR² mass spectrometry dataset. For more advanced tasks, users can repeat these steps, leveraging other record sets and fields by `@id` for deeper domain-specific analyses.*